# All ML and QML experiments — notebook control centre

Run this notebook from top to bottom for an overview. It covers the original five-stock study, binary/three-class extension, scaling, temporal robustness, expanded models, and the final equal-sample 20-stock benchmark. No terminal commands are needed. For fully notebook-contained experiment code, use notebooks 09–13; this control centre can also launch the existing implementations.

The default switches load existing evidence quickly. Turn on only the experiment you intend to rerun; quantum kernels can be expensive. IBM hardware is intentionally excluded.


## 1. Setup and run switches

Use the project's `.venv-stage4` Python environment as the Jupyter kernel. Set a switch to `True` to regenerate that experiment. The first five original notebooks remain the detailed, runnable original study; this notebook incorporates their saved evidence and runs every later experiment.


In [ ]:
from pathlib import Path
import json, runpy
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name.lower() == 'notebooks':
    ROOT = ROOT.parent
assert (ROOT / 'experiments' / 'complete_benchmark.py').exists(), f'Open Jupyter in the project folder, not {ROOT}'
RUN_EXTENSION = False
RUN_SCALING = False
RUN_TEMPORAL = False
RUN_EXPANDED = False
INCLUDE_PROJECTED_KERNEL = False
RUN_FINAL_BINARY = False
RUN_FINAL_THREE_CLASS = False
RUN_FINAL_VERIFICATION = False  # set True to recompute the full paired bootstrap audit
print('Project:', ROOT)
print('Only enabled experiments will be regenerated.')

## 2. Original five-stock experiments

The original workflow is already fully notebook-based: `01_data_understanding.ipynb`, `02_feature_engineering.ipynb`, `03_classical_svm.ipynb`, `04_qsvm.ipynb`, and `05_comparison_analysis.ipynb`. Run these in order if regenerating the original study. This cell reads its saved metrics without rerunning it. The original and final 20-stock studies have different sample sizes and should not be mixed into one leaderboard.


In [ ]:
original_paths = {'classical_stage3': ROOT/'results/stage3/metrics.csv', 'quantum_stage4': ROOT/'results/stage4/metrics.csv'}
for name, path in original_paths.items():
    if path.exists():
        table = pd.read_csv(path)
        print(f'{name}: {len(table)} model rows')
        display(table.head())
    else:
        print(f'{name}: not run yet; use notebooks 03–04.')

## 3. Binary and three-class extension

Five fixed stocks, chronological selection and evaluation, two- and four-qubit quantum kernels. Turning on this switch runs both classical and QML stages for both classification tasks. Saved outputs under `results/extension` are exploratory.


In [ ]:
if RUN_EXTENSION:
    extension = runpy.run_path(str(ROOT/'experiments/multiclass_extension.py'))
    for stage in (3, 4):
        print(f'Running extension stage {stage}...')
        extension['run'](ROOT, stage)
for stage in (3, 4):
    path = ROOT/f'results/extension/stage{stage}/metrics.csv'
    if path.exists():
        table = pd.read_csv(path)
        print(f'Extension stage {stage}: {len(table)} rows')
        display(table[['task','dimension','model','balanced_accuracy','validation_balanced_accuracy']].sort_values(['task','dimension','balanced_accuracy'], ascending=[True,True,False]))
    else:
        print(f'Extension stage {stage}: enable RUN_EXTENSION to create results.')

## 4. Development-set scaling experiment

Compares the prior 1,400-row development protocol to the 2,000-row protocol. Both tasks and both classical/QML stages are included. These are **not** the same sample size as the final 20-stock matched benchmark.


In [ ]:
if RUN_SCALING:
    scaling = runpy.run_path(str(ROOT/'experiments/scaling_experiment.py'))
    for stage in (3, 4):
        print(f'Running scaling stage {stage}...')
        scaling['run'](ROOT, stage)
scale_rows = []
for size, base in [(1400, ROOT/'results/extension'), (2000, ROOT/'results/scaling/development_2000')]:
    for stage in (3, 4):
        path = base/f'stage{stage}/metrics.csv'
        if path.exists():
            part = pd.read_csv(path).assign(development_rows=size, stage=stage)
            scale_rows.append(part)
if scale_rows:
    scale_table = pd.concat(scale_rows, ignore_index=True)
    display(scale_table[['development_rows','stage','task','dimension','model','balanced_accuracy']].sort_values(['task','dimension','model','development_rows']))
else:
    print('No scaling outputs yet; enable RUN_EXTENSION and RUN_SCALING.')

## 5. Temporal robustness experiment

Moves the development window to 2021–2022 and compares it with the original extension window. Both stages and both tasks are run by the switch. A single changed window is a robustness check, not an independent prospective validation.


In [ ]:
if RUN_TEMPORAL:
    temporal = runpy.run_path(str(ROOT/'experiments/temporal_robustness.py'))
    for stage in (3, 4):
        print(f'Running temporal stage {stage}...')
        temporal['run'](ROOT, stage)
temporal_rows = []
for window, base in [('original', ROOT/'results/extension'), ('2021–2022', ROOT/'results/temporal_robustness/window_2021_2022')]:
    for stage in (3, 4):
        path = base/f'stage{stage}/metrics.csv'
        if path.exists():
            temporal_rows.append(pd.read_csv(path).assign(window=window, stage=stage))
if temporal_rows:
    temporal_table = pd.concat(temporal_rows, ignore_index=True)
    display(temporal_table[['window','stage','task','dimension','model','balanced_accuracy']].sort_values(['task','dimension','model','window']))
else:
    print('No temporal outputs yet; enable RUN_TEMPORAL.')

## 6. Expanded matched models

Adds logistic regression, random forest, gradient boosting, k-nearest neighbours, MLP, and quantum-kernel variants to the five-stock matched study. The projected quantum kernel is optional and slower. The detailed report is `07_expanded_comparison.ipynb`; this cell provides a one-switch notebook run and a leaderboard.


In [ ]:
if RUN_EXPANDED:
    import sys
    sys.path.insert(0, str(ROOT/'experiments'))
    import expanded_comparison as expanded
    expanded.run(ROOT, projected=INCLUDE_PROJECTED_KERNEL)
expanded_path = ROOT/'results/expanded/metrics.csv'
if expanded_path.exists():
    expanded_table = pd.read_csv(expanded_path)
    display(expanded_table.sort_values(['task','dimension','balanced_accuracy'], ascending=[True,True,False]))
else:
    print('Expanded comparison not yet run. Enable RUN_EXPANDED, or use notebook 07.')

## 7. Final equal-sample 20-stock ML–QML benchmark

Five classical models, one dummy baseline, and four quantum-kernel SVM variants for both binary and three-class tasks. Every model in a task sees the same sampled 1,599 training, 800 validation, 1,200 test and 1,200 robustness stock-date rows. The benchmark is simulator-only, not IBM hardware. The detailed interpretation is `06_ml_qml_comparison.ipynb`.

Rerunning this section overwrites the saved benchmark evidence and may take substantial time. Keep the switches off for a quick review.


In [ ]:
if RUN_FINAL_BINARY or RUN_FINAL_THREE_CLASS:
    benchmark = runpy.run_path(str(ROOT/'experiments/complete_benchmark.py'))
    for task, enabled in [('binary',RUN_FINAL_BINARY), ('three_class',RUN_FINAL_THREE_CLASS)]:
        if enabled:
            print(f'Running final {task} benchmark...')
            benchmark['run'](ROOT, task)
final_tables = {}
for task in ('binary','three_class'):
    path = ROOT/f'results/complete_benchmark/{task}_metrics.csv'
    if path.exists():
        final_tables[task] = pd.read_csv(path)
        print(f'{task}: {len(final_tables[task])} models')
        display(final_tables[task][['model','validation_balanced_accuracy','test_balanced_accuracy','confirmatory_balanced_accuracy','fit_seconds']].sort_values('test_balanced_accuracy', ascending=False))
    else:
        print(f'{task}: enable RUN_FINAL_{task.upper()} to create results.')

## 8. Verify equality and recompute metrics

The default loads the saved audit. Set `RUN_FINAL_VERIFICATION=True` to recompute each model's test/robustness scores from predictions, check identical stock-date rows and labels, and rerun paired date-block bootstrap intervals against the validation-selected classical model. Recomputing the bootstrap can take minutes. It does **not** correct for choosing the best among several QML variants, so treat conclusions as exploratory.


In [ ]:
verification = runpy.run_path(str(ROOT/'experiments/verify_comparison.py'))
for task in ('binary','three_class'):
    base = ROOT/'results/complete_benchmark'
    if not (base/f'{task}_predictions.csv').exists():
        print(f'{task}: no predictions to verify')
        continue
    report_path = base/f'{task}_verification.json'
    report = verification['verify_task'](base, task) if RUN_FINAL_VERIFICATION or not report_path.exists() else json.loads(report_path.read_text(encoding='utf-8'))
    print(f"{task}: {report['verified_metric_values']} metrics checked; {report['matched_test_rows']} identical test rows; {report['matched_test_dates']} test dates")
    print('Validation-selected classical comparator:', report['validation_selected_classical'])
    display(pd.DataFrame.from_dict(report['paired_date_block_comparisons'], orient='index'))

## 9. Interpretation and next step

Compare models *within* each experiment, never by merging these leaderboards across different sample sizes or time windows. The strongest matched test comparison is in section 7 and notebook 06. Treat 2025-onward and the changed-window analyses as previously inspected robustness evidence. For a more trustworthy claim, freeze this protocol and collect a genuinely untouched future period, then rerun this notebook. IBM Quantum hardware setup remains for later.
